# Fine-Grained Segmentation Model - Evaluation & Analysis

This notebook provides comprehensive evaluation and analysis of the trained segmentation model.

## Analysis Components:
1. **Model Loading**: Load the best trained checkpoint
2. **Test Set Evaluation**: Complete metrics on test data
3. **Class-wise Analysis**: Per-class IoU, precision, recall, F1-score
4. **Confusion Matrix**: Detailed class confusion analysis
5. **Visual Analysis**: Sample predictions with overlays
6. **Error Analysis**: Identify challenging samples
7. **Detailed Report**: Export comprehensive metrics

## Metrics Reported:
- Mean IoU (Intersection over Union)
- Per-class IoU
- Pixel Accuracy
- Per-class Precision, Recall, F1-Score
- Confusion Matrix
- Boundary IoU (for edge quality)

## 1. Setup and Imports

In [ ]:
# Install dependencies if needed
!pip install -q segmentation-models-pytorch albumentations timm scikit-learn seaborn

print("✓ Dependencies installed!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

import os
from pathlib import Path
from tqdm.auto import tqdm
import json
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Paths
DATASET_ROOT = "/kaggle/input/datasets/muhammadannasshaikh/dsscomp"
TEST_DIR = os.path.join(DATASET_ROOT, "test_public_80")

# Model checkpoint path - UPDATE THIS to your checkpoint location
CHECKPOINT_PATH = "/kaggle/working/checkpoints/model_best.pth"
# Alternative: "/kaggle/input/your-model-dataset/model_best.pth" if uploaded as dataset

# Output directories
OUTPUT_DIR = "/kaggle/working"
ANALYSIS_DIR = os.path.join(OUTPUT_DIR, "analysis")
VISUALIZATIONS_DIR = os.path.join(ANALYSIS_DIR, "visualizations")
REPORTS_DIR = os.path.join(ANALYSIS_DIR, "reports")

os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(VISUALIZATIONS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Model configuration (must match training)
IMAGE_SIZE = (544, 960)  # Height x Width
BATCH_SIZE = 4
ENCODER = 'timm-efficientnet-b5'

# Class information
VALUE_MAP = {
    0: 0,        # background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    700: 6,      # Logs
    800: 7,      # Rocks
    7100: 8,     # Landscape
    10000: 9     # Sky
}
N_CLASSES = len(VALUE_MAP)

CLASS_NAMES = [
    'Background', 'Trees', 'Lush Bushes', 'Dry Grass', 'Dry Bushes',
    'Ground Clutter', 'Logs', 'Rocks', 'Landscape', 'Sky'
]

# Color mapping for visualization
CLASS_COLORS = np.array([
    [0, 0, 0],       # Background - Black
    [0, 128, 0],     # Trees - Green
    [0, 255, 0],     # Lush Bushes - Bright Green
    [255, 255, 0],   # Dry Grass - Yellow
    [139, 69, 19],   # Dry Bushes - Brown
    [160, 82, 45],   # Ground Clutter - Sienna
    [101, 67, 33],   # Logs - Dark Brown
    [128, 128, 128], # Rocks - Gray
    [210, 180, 140], # Landscape - Tan
    [135, 206, 235], # Sky - Sky Blue
], dtype=np.uint8)

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Test directory: {TEST_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Number of classes: {N_CLASSES}")
print(f"Output directory: {ANALYSIS_DIR}")
print("="*80)

## 3. Dataset Class

In [ ]:
def convert_mask(mask_array):
    """Convert raw mask values to class IDs."""
    new_mask = np.zeros_like(mask_array, dtype=np.uint8)
    for raw_value, class_id in VALUE_MAP.items():
        new_mask[mask_array == raw_value] = class_id
    return new_mask


def get_validation_augmentation():
    """Only normalization for test."""
    return A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


class SegmentationDataset(Dataset):
    """Segmentation dataset for evaluation."""
    
    def __init__(self, data_dir, image_size=(544, 960), augmentation=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.mask_dir = os.path.join(data_dir, 'Segmentation')
        self.image_size = image_size
        self.augmentation = augmentation
        self.image_files = sorted(os.listdir(self.image_dir))
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_image = image.copy()  # Keep for visualization
        image = cv2.resize(image, (self.image_size[1], self.image_size[0]), 
                          interpolation=cv2.INTER_LINEAR)
        
        # Load mask
        mask_path = os.path.join(self.mask_dir, img_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.image_size[1], self.image_size[0]), 
                         interpolation=cv2.INTER_NEAREST)
        mask = convert_mask(mask)
        
        # Apply augmentation
        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']
            return {
                'image': image,
                'mask': mask.long(),
                'original_image': original_image,
                'filename': img_name
            }
        else:
            return {
                'image': torch.from_numpy(image).permute(2, 0, 1).float(),
                'mask': torch.from_numpy(mask).long(),
                'original_image': original_image,
                'filename': img_name
            }

print("✓ Dataset class created!")

## 4. Load Model

In [ ]:
class FineGrainedSegmentationModel(nn.Module):
    """High-resolution segmentation model."""
    
    def __init__(self, encoder_name, num_classes):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=None,  # We'll load from checkpoint
            in_channels=3,
            classes=num_classes,
            activation=None,
        )
        
    def forward(self, x):
        return self.model(x)


# Create model
model = FineGrainedSegmentationModel(
    encoder_name=ENCODER,
    num_classes=N_CLASSES
)

# Load checkpoint
print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("\n" + "="*80)
print("MODEL LOADED SUCCESSFULLY")
print("="*80)
if 'epoch' in checkpoint:
    print(f"Checkpoint epoch: {checkpoint['epoch'] + 1}")
if 'train_iou' in checkpoint:
    print(f"Training IoU: {checkpoint['train_iou']:.4f}")
if 'test_iou' in checkpoint:
    print(f"Test IoU: {checkpoint['test_iou']:.4f}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print("="*80)

## 5. Load Test Dataset

In [ ]:
test_dataset = SegmentationDataset(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    augmentation=get_validation_augmentation()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Test dataset: {len(test_dataset)} samples")
print(f"Test batches: {len(test_loader)}")

## 6. Comprehensive Metrics Functions

In [ ]:
def compute_iou_per_class(pred, target, num_classes=N_CLASSES):
    """Compute IoU for each class separately."""
    pred = pred.view(-1)
    target = target.view(-1)
    
    iou_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        
        if union == 0:
            iou_per_class.append(float('nan'))
        else:
            iou_per_class.append((intersection / union).item())
    
    return iou_per_class


def compute_precision_recall_f1(pred, target, num_classes=N_CLASSES):
    """Compute precision, recall, F1 for each class."""
    pred = pred.view(-1)
    target = target.view(-1)
    
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        
        tp = (pred_inds & target_inds).sum().float()
        fp = (pred_inds & ~target_inds).sum().float()
        fn = (~pred_inds & target_inds).sum().float()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = float('nan')
        
        precision_per_class.append(precision.item() if torch.is_tensor(precision) else precision)
        recall_per_class.append(recall.item() if torch.is_tensor(recall) else recall)
        f1_per_class.append(f1.item() if torch.is_tensor(f1) else f1)
    
    return precision_per_class, recall_per_class, f1_per_class


def compute_boundary_iou(pred, target, dilation=2):
    """Compute IoU on boundary pixels only."""
    pred_np = pred.cpu().numpy()
    target_np = target.cpu().numpy()
    
    # Find boundaries using morphological operations
    kernel = np.ones((dilation, dilation), np.uint8)
    
    pred_dilated = cv2.dilate(pred_np.astype(np.uint8), kernel)
    pred_boundary = (pred_dilated != pred_np)
    
    target_dilated = cv2.dilate(target_np.astype(np.uint8), kernel)
    target_boundary = (target_dilated != target_np)
    
    # Compute IoU on boundary pixels
    boundary_mask = pred_boundary | target_boundary
    if boundary_mask.sum() == 0:
        return float('nan')
    
    pred_boundary_masked = pred_np[boundary_mask]
    target_boundary_masked = target_np[boundary_mask]
    
    intersection = (pred_boundary_masked == target_boundary_masked).sum()
    union = len(pred_boundary_masked)
    
    return intersection / union if union > 0 else float('nan')


print("✓ Metrics functions defined!")

## 7. Run Evaluation on Test Set

In [ ]:
print("\n" + "="*80)
print("RUNNING EVALUATION ON TEST SET")
print("="*80)

# Initialize accumulators
all_predictions = []
all_targets = []
all_iou_per_class = {i: [] for i in range(N_CLASSES)}
all_precision = {i: [] for i in range(N_CLASSES)}
all_recall = {i: [] for i in range(N_CLASSES)}
all_f1 = {i: [] for i in range(N_CLASSES)}
all_boundary_iou = []
pixel_accuracies = []

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        # Forward pass
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)
        
        # Store predictions and targets
        all_predictions.append(predictions.cpu())
        all_targets.append(masks.cpu())
        
        # Compute per-sample metrics
        for i in range(predictions.shape[0]):
            pred = predictions[i]
            target = masks[i]
            
            # IoU per class
            iou_per_class = compute_iou_per_class(pred, target)
            for class_id, iou in enumerate(iou_per_class):
                if not np.isnan(iou):
                    all_iou_per_class[class_id].append(iou)
            
            # Precision, Recall, F1
            precision, recall, f1 = compute_precision_recall_f1(pred, target)
            for class_id in range(N_CLASSES):
                if not np.isnan(precision[class_id]):
                    all_precision[class_id].append(precision[class_id])
                if not np.isnan(recall[class_id]):
                    all_recall[class_id].append(recall[class_id])
                if not np.isnan(f1[class_id]):
                    all_f1[class_id].append(f1[class_id])
            
            # Boundary IoU
            boundary_iou = compute_boundary_iou(pred, target)
            if not np.isnan(boundary_iou):
                all_boundary_iou.append(boundary_iou)
            
            # Pixel accuracy
            pixel_acc = (pred == target).float().mean().item()
            pixel_accuracies.append(pixel_acc)

# Concatenate all predictions and targets
all_predictions = torch.cat(all_predictions, dim=0)
all_targets = torch.cat(all_targets, dim=0)

print("\n✓ Evaluation complete!")

## 8. Compute and Display Overall Metrics

In [ ]:
# Compute overall metrics
mean_iou_per_class = {i: np.mean(all_iou_per_class[i]) if all_iou_per_class[i] else 0.0 
                       for i in range(N_CLASSES)}
mean_precision = {i: np.mean(all_precision[i]) if all_precision[i] else 0.0 
                  for i in range(N_CLASSES)}
mean_recall = {i: np.mean(all_recall[i]) if all_recall[i] else 0.0 
               for i in range(N_CLASSES)}
mean_f1 = {i: np.mean(all_f1[i]) if all_f1[i] else 0.0 
           for i in range(N_CLASSES)}

overall_miou = np.mean([v for v in mean_iou_per_class.values() if v > 0])
overall_pixel_acc = np.mean(pixel_accuracies)
overall_boundary_iou = np.mean(all_boundary_iou) if all_boundary_iou else 0.0

# Display results
print("\n" + "="*80)
print("OVERALL METRICS")
print("="*80)
print(f"Mean IoU: {overall_miou:.4f}")
print(f"Pixel Accuracy: {overall_pixel_acc:.4f}")
print(f"Boundary IoU: {overall_boundary_iou:.4f}")
print("="*80)

print("\n" + "="*80)
print("CLASS-WISE METRICS")
print("="*80)
print(f"{'Class':<20} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1-Score':>10}")
print("-" * 80)
for i in range(N_CLASSES):
    print(f"{CLASS_NAMES[i]:<20} {mean_iou_per_class[i]:>8.4f} {mean_precision[i]:>10.4f} "
          f"{mean_recall[i]:>8.4f} {mean_f1[i]:>10.4f}")
print("="*80)

## 9. Create Class-wise Metrics Bar Chart

In [ ]:
# Create DataFrame for metrics
metrics_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'IoU': [mean_iou_per_class[i] for i in range(N_CLASSES)],
    'Precision': [mean_precision[i] for i in range(N_CLASSES)],
    'Recall': [mean_recall[i] for i in range(N_CLASSES)],
    'F1-Score': [mean_f1[i] for i in range(N_CLASSES)]
})

# Plot class-wise metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# IoU
axes[0, 0].barh(metrics_df['Class'], metrics_df['IoU'], color='steelblue')
axes[0, 0].set_xlabel('IoU', fontsize=12)
axes[0, 0].set_title('Class-wise IoU', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].grid(axis='x', alpha=0.3)

# Precision
axes[0, 1].barh(metrics_df['Class'], metrics_df['Precision'], color='green')
axes[0, 1].set_xlabel('Precision', fontsize=12)
axes[0, 1].set_title('Class-wise Precision', fontsize=14, fontweight='bold')
axes[0, 1].set_xlim(0, 1)
axes[0, 1].grid(axis='x', alpha=0.3)

# Recall
axes[1, 0].barh(metrics_df['Class'], metrics_df['Recall'], color='orange')
axes[1, 0].set_xlabel('Recall', fontsize=12)
axes[1, 0].set_title('Class-wise Recall', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim(0, 1)
axes[1, 0].grid(axis='x', alpha=0.3)

# F1-Score
axes[1, 1].barh(metrics_df['Class'], metrics_df['F1-Score'], color='red')
axes[1, 1].set_xlabel('F1-Score', fontsize=12)
axes[1, 1].set_title('Class-wise F1-Score', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'classwise_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'classwise_metrics.png')}")

## 10. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(
    all_targets.view(-1).numpy(), 
    all_predictions.view(-1).numpy(),
    labels=list(range(N_CLASSES))
)

# Normalize confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)  # Replace NaN with 0

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'confusion_matrix.png')}")

## 11. Sample Predictions Visualization

In [ ]:
# Select diverse samples for visualization
num_samples = 8
sample_indices = np.linspace(0, len(test_dataset)-1, num_samples, dtype=int)

fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 3))

model.eval()
with torch.no_grad():
    for idx, sample_idx in enumerate(sample_indices):
        sample = test_dataset[sample_idx]
        image = sample['image'].unsqueeze(0).to(device)
        mask = sample['mask'].numpy()
        original_img = sample['original_image']
        
        # Get prediction
        output = model(image)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]
        
        # Denormalize image
        img_display = image.cpu().numpy()[0].transpose(1, 2, 0)
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)
        
        # Create overlay
        pred_colored = CLASS_COLORS[pred]
        overlay = cv2.addWeighted(
            (img_display * 255).astype(np.uint8), 0.6,
            pred_colored, 0.4,
            0
        )
        
        # Plot
        axes[idx, 0].imshow(img_display)
        axes[idx, 0].set_title(f'Input Image ({sample["filename"]})', fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(CLASS_COLORS[mask])
        axes[idx, 1].set_title('Ground Truth', fontsize=10)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(pred_colored)
        axes[idx, 2].set_title('Prediction', fontsize=10)
        axes[idx, 2].axis('off')
        
        axes[idx, 3].imshow(overlay)
        axes[idx, 3].set_title('Overlay', fontsize=10)
        axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'sample_predictions.png')}")

## 12. Error Analysis - Best and Worst Predictions

In [ ]:
# Compute IoU for each test sample
sample_ious = []
model.eval()
with torch.no_grad():
    for i in range(len(test_dataset)):
        sample = test_dataset[i]
        image = sample['image'].unsqueeze(0).to(device)
        mask = sample['mask']
        
        output = model(image)
        pred = torch.argmax(output, dim=1).cpu()[0]
        
        iou = np.nanmean(compute_iou_per_class(pred, mask))
        sample_ious.append((i, iou, sample['filename']))

# Sort by IoU
sample_ious.sort(key=lambda x: x[1])

# Get best and worst samples
worst_samples = sample_ious[:4]  # 4 worst
best_samples = sample_ious[-4:]  # 4 best

# Visualize worst predictions
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
fig.suptitle('Worst Predictions (Lowest IoU)', fontsize=16, fontweight='bold')

with torch.no_grad():
    for idx, (sample_idx, iou, filename) in enumerate(worst_samples):
        sample = test_dataset[sample_idx]
        image = sample['image'].unsqueeze(0).to(device)
        mask = sample['mask'].numpy()
        
        output = model(image)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]
        
        img_display = image.cpu().numpy()[0].transpose(1, 2, 0)
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)
        
        axes[idx, 0].imshow(img_display)
        axes[idx, 0].set_title(f'{filename}\nIoU: {iou:.3f}', fontsize=9)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(CLASS_COLORS[mask])
        axes[idx, 1].set_title('Ground Truth', fontsize=9)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(CLASS_COLORS[pred])
        axes[idx, 2].set_title('Prediction', fontsize=9)
        axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'worst_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

# Visualize best predictions
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
fig.suptitle('Best Predictions (Highest IoU)', fontsize=16, fontweight='bold')

with torch.no_grad():
    for idx, (sample_idx, iou, filename) in enumerate(best_samples):
        sample = test_dataset[sample_idx]
        image = sample['image'].unsqueeze(0).to(device)
        mask = sample['mask'].numpy()
        
        output = model(image)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]
        
        img_display = image.cpu().numpy()[0].transpose(1, 2, 0)
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)
        
        axes[idx, 0].imshow(img_display)
        axes[idx, 0].set_title(f'{filename}\nIoU: {iou:.3f}', fontsize=9)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(CLASS_COLORS[mask])
        axes[idx, 1].set_title('Ground Truth', fontsize=9)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(CLASS_COLORS[pred])
        axes[idx, 2].set_title('Prediction', fontsize=9)
        axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'best_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'worst_predictions.png')}")
print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'best_predictions.png')}")

## 13. IoU Distribution Analysis

In [ ]:
# Plot IoU distribution
ious_only = [iou for _, iou, _ in sample_ious]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ious_only, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(ious_only), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(ious_only):.3f}')
axes[0].axvline(np.median(ious_only), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(ious_only):.3f}')
axes[0].set_xlabel('IoU', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('IoU Distribution Across Test Set', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(ious_only, vert=True)
axes[1].set_ylabel('IoU', fontsize=12)
axes[1].set_title('IoU Distribution (Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'iou_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'iou_distribution.png')}")
print(f"\nIoU Statistics:")
print(f"  Mean: {np.mean(ious_only):.4f}")
print(f"  Median: {np.median(ious_only):.4f}")
print(f"  Std: {np.std(ious_only):.4f}")
print(f"  Min: {np.min(ious_only):.4f}")
print(f"  Max: {np.max(ious_only):.4f}")

## 14. Class Frequency Analysis

In [ ]:
# Count class frequencies in ground truth and predictions
gt_class_counts = np.bincount(all_targets.view(-1).numpy(), minlength=N_CLASSES)
pred_class_counts = np.bincount(all_predictions.view(-1).numpy(), minlength=N_CLASSES)

# Create comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(N_CLASSES)
width = 0.35

# Bar chart
axes[0].bar(x - width/2, gt_class_counts, width, label='Ground Truth', alpha=0.8)
axes[0].bar(x + width/2, pred_class_counts, width, label='Predictions', alpha=0.8)
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Pixel Count', fontsize=12)
axes[0].set_title('Class Frequency: Ground Truth vs Predictions', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Percentage difference
pct_diff = ((pred_class_counts - gt_class_counts) / (gt_class_counts + 1e-6)) * 100
colors = ['green' if x < 0 else 'red' for x in pct_diff]
axes[1].barh(CLASS_NAMES, pct_diff, color=colors, alpha=0.7)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Percentage Difference (%)', fontsize=12)
axes[1].set_title('Prediction Bias: Over/Under Prediction', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'class_frequency_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'class_frequency_analysis.png')}")

## 15. Export Detailed Report

In [ ]:
# Create comprehensive report
report = {
    'model_info': {
        'architecture': 'UNet++ with EfficientNet-B5',
        'encoder': ENCODER,
        'image_size': IMAGE_SIZE,
        'num_classes': N_CLASSES,
        'checkpoint': CHECKPOINT_PATH,
        'total_parameters': sum(p.numel() for p in model.parameters())
    },
    'overall_metrics': {
        'mean_iou': float(overall_miou),
        'pixel_accuracy': float(overall_pixel_acc),
        'boundary_iou': float(overall_boundary_iou),
    },
    'classwise_metrics': {},
    'iou_statistics': {
        'mean': float(np.mean(ious_only)),
        'median': float(np.median(ious_only)),
        'std': float(np.std(ious_only)),
        'min': float(np.min(ious_only)),
        'max': float(np.max(ious_only)),
        'q1': float(np.percentile(ious_only, 25)),
        'q3': float(np.percentile(ious_only, 75))
    },
    'test_set_info': {
        'num_samples': len(test_dataset),
        'total_pixels': int(all_targets.numel())
    }
}

# Add classwise metrics
for i in range(N_CLASSES):
    report['classwise_metrics'][CLASS_NAMES[i]] = {
        'iou': float(mean_iou_per_class[i]),
        'precision': float(mean_precision[i]),
        'recall': float(mean_recall[i]),
        'f1_score': float(mean_f1[i]),
        'ground_truth_pixels': int(gt_class_counts[i]),
        'predicted_pixels': int(pred_class_counts[i])
    }

# Save as JSON
report_path = os.path.join(REPORTS_DIR, 'evaluation_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✓ Saved: {report_path}")

# Save metrics DataFrame as CSV
metrics_df.to_csv(os.path.join(REPORTS_DIR, 'classwise_metrics.csv'), index=False)
print(f"✓ Saved: {os.path.join(REPORTS_DIR, 'classwise_metrics.csv')}")

# Save confusion matrix
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(os.path.join(REPORTS_DIR, 'confusion_matrix.csv'))
print(f"✓ Saved: {os.path.join(REPORTS_DIR, 'confusion_matrix.csv')}")

## 16. Create Summary Legend

In [ ]:
# Create class color legend
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# Create patches for legend
patches = []
for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    rgb_color = tuple(c/255.0 for c in color)
    patch = mpatches.Patch(color=rgb_color, label=f'{name} (IoU: {mean_iou_per_class[i]:.3f})')
    patches.append(patch)

ax.legend(handles=patches, loc='center', fontsize=12, title='Class Color Legend', title_fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATIONS_DIR, 'class_legend.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(VISUALIZATIONS_DIR, 'class_legend.png')}")

## 17. Final Summary

In [ ]:
print("\n" + "="*80)
print("EVALUATION COMPLETE - SUMMARY")
print("="*80)
print(f"\nTest Samples: {len(test_dataset)}")
print(f"\nOverall Performance:")
print(f"  Mean IoU: {overall_miou:.4f}")
print(f"  Pixel Accuracy: {overall_pixel_acc:.4f}")
print(f"  Boundary IoU: {overall_boundary_iou:.4f}")

print(f"\nTop 3 Performing Classes:")
sorted_classes = sorted(mean_iou_per_class.items(), key=lambda x: x[1], reverse=True)
for i, (class_id, iou) in enumerate(sorted_classes[:3], 1):
    print(f"  {i}. {CLASS_NAMES[class_id]}: {iou:.4f}")

print(f"\nBottom 3 Performing Classes:")
for i, (class_id, iou) in enumerate(sorted_classes[-3:], 1):
    print(f"  {i}. {CLASS_NAMES[class_id]}: {iou:.4f}")

print(f"\nOutput Directories:")
print(f"  Visualizations: {VISUALIZATIONS_DIR}")
print(f"  Reports: {REPORTS_DIR}")

print(f"\nGenerated Files:")
print(f"  - classwise_metrics.png")
print(f"  - confusion_matrix.png")
print(f"  - sample_predictions.png")
print(f"  - worst_predictions.png")
print(f"  - best_predictions.png")
print(f"  - iou_distribution.png")
print(f"  - class_frequency_analysis.png")
print(f"  - class_legend.png")
print(f"  - evaluation_report.json")
print(f"  - classwise_metrics.csv")
print(f"  - confusion_matrix.csv")
print("="*80)